In [57]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [58]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [59]:
dim_proveedor=pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero=pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede=pd.read_sql_table('dim_sede', etl_conn)
dim_hora=pd.read_sql_table('dim_hora', etl_conn)    
dim_geografia=pd.read_sql_table('dim_geografia', etl_conn)
dim_fecha=pd.read_sql_table('dim_fecha', etl_conn)
dim_usuario=pd.read_sql_table('dim_usuario', etl_conn)
trans_servicio=pd.read_sql_table('trans_servicio', etl_conn)

tabla_sede=pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)

In [60]:
trans_servicio.head()


,servicio_id,fecha_solicitud,hora_solicitud,cliente_id,mensajero_id,usuario_id,ciudad_destino_id,ciudad_origen_id,mensajero2_id,mensajero3_id,...,fecha_iniciado,fecha_mensajero_asignado,fecha_recogido_mensajero,fecha_entregado,fecha_finalizado_completo,hora_iniciado,hora_mensajero_asignado,hora_recogido_mensajero,hora_entregado,hora_finalizado_completo
0,510,2024-02-03,09:37:18,5,0,7,1,1,NaN,NaN,...,2024-02-03,2024-02-03,NaT,NaT,NaT,09:37:18,10:00:26,None,None,None
1,794,2024-02-08,14:15:40,5,0,7,1,1,NaN,NaN,...,2024-02-08,NaT,NaT,NaT,NaT,14:15:40,None,None,None,None
2,791,2024-02-08,12:07:55,5,0,11,1,1,NaN,NaN,...,2024-02-08,NaT,NaT,NaT,NaT,12:07:55,None,None,None,None
3,154,2024-01-25,08:45:15,5,0,7,1,1,NaN,NaN,...,2024-01-25,2024-01-25,NaT,NaT,NaT,08:45:15,10:48:39,None,None,None
4,16686,2024-06-12,09:00:31,11,0,160,1,6,NaN,NaN,...,2024-06-12,2024-06-12,2024-06-12,2024-06-13,2024-06-13,09:00:31,09:03:20,16:36:31,09:09:34,09:10:01


In [ ]:
hecho_servicios= trans_servicio.merge(dim_proveedor[['id_proveedor','key_proveedor']], left_on='cliente_id',right_on='id_proveedor', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero_id',right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero2_id', right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero3_id', right_on='id_operaciones', how='left')  \
    .merge(dim_sede[['sede_id','key_sede']], left_on='sede_id', right_on='sede_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_origen_id',right_on='ciudad_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_destino_id',right_on='ciudad_id', how='left') \
    


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_solicitud', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora','key_hora']], left_on='hora_solicitud', right_on='hora', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_solicitud','key_hora':'key_dim_hora_solicitud'}, inplace=False)
                            
                            
hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_iniciado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_iniciado', right_on='hora_formato', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_iniciado','key_hora':'key_dim_hora_iniciado'}, inplace=False)


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_mensajero_asignado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_mensajero_asignado', right_on='hora_formato', how='left') \

hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_mensajero_asignado','key_hora':'key_dim_hora_mensajero_asignado'}, inplace=False)
#Queda haciendo falta arreglar los merge y ya los duplicados generan error

hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_recogido_mensajero', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_recogido_mensajero', right_on='hora_formato', how='left') \

hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_recogido_mensajero','key_hora':'key_dim_hora_recogido_mensajero'}, inplace=False)
                            


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_entregado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_entregado', right_on='hora_formato', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_entregado','key_hora':'key_dim_hora_entregado'}, inplace=False)


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_finalizado_completo', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_finalizado_completo', right_on='hora_formato', how='left') \

hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_finalizado_completo','key_hora':'key_dim_hora_finalizado_completo'}, inplace=True)

hecho_servicios.drop(columns=['id_proveedor','id_operaciones_x','id_operaciones_y','id_operaciones','ciudad_id_x','ciudad_id_y',
                              'mensajero_id','mensajero2_id','mensajero3_id','ciudad_origen_id','ciudad_destino_id','sede_id','cliente_id','usuario_id',
                                'user_id'
                              ], inplace=True)
hecho_servicios.rename(columns={'key_proveedor':'key_dim_proveedor',
                                'key_mensajero_x':'key_dim_mensajero',
                                'key_mensajero_y':'key_dim_mensajero2',
                                'key_mensajero':'key_dim_mensajero3',
                                'key_geografia_x':'key_dim_geografia_origen',
                                'key_geografia_y':'key_dim_geografia_destino'
                                }, inplace=True)

hecho_servicios.info()

MergeError: Passing 'suffixes' which cause duplicate columns {'fecha_completa_x'} is not allowed.

In [ ]:
tabla_estados.head(10)

,id,fecha,hora,foto,observaciones,estado_id,servicio_id,es_prueba,foto_binary
0,1014,2024-01-29,01:13:32,foto,4 tubos,4,226,False,None
1,1484,2024-01-30,18:45:12,foto,Demora,3,79,True,None
2,2829,2024-02-06,11:34:04,foto,Compra exitosa,5,613,False,None
3,1888,2024-02-01,14:50:39,foto,Zzxzz,4,376,False,None
4,32312,2024-04-06,16:11:21,foto,No,3,7164,True,None
5,2426,2024-02-04,11:15:40,foto,Vxvx,3,377,True,None
6,48803,2024-05-03,06:11:39,foto,",",3,10910,True,None
7,3323,2024-02-07,16:26:03,foto,Con mensajero Asignado,2,746,True,None
8,3886,2024-02-09,13:25:05,foto,Recibido,6,842,False,None
9,4211,2024-02-10,13:46:43,foto,Con mensajero Asignado,2,930,True,None
